# W Loading Tests — Factor Loading Recovery on Synthetic Data

Cztery sanity-checki weryfikujące, czy model poprawnie odzyskuje macierz ładunków czynnikowych `W`.

| # | Cel | W_prior | Z_prior | Co testuje |
|---|-----|---------|---------|------------|
| **SC1** | Podstawowe odtworzenie `W` z losowych danych (sweep po szumie) | ARD | STD_NORMAL | Czy `W_est ≈ W_true` w warunkach bez kohort, jak spada jakość ze szumem |
| **SC2** | Rzadka struktura blokowa `W` | ARD_SS | STD_NORMAL | Czy spike-and-slab `E[S]` poprawnie wskazuje niezerowe ładunki |
| **SC3** | Cecha różnicująca kohorty | ARD_SS | COHORT | Czy unikalny ładunek dla jednej cechy jest poprawnie zlokalizowany, gdy ten czynnik jest kohortowy |
| **SC4** | Odporność `W` na sygnał kohortowy | ARD_SS | COHORT vs STD_NORMAL | Czy model kohortowy zachowuje jakość `W` przy rosnącym `μ` kohort |

Każda sekcja zawiera:
1. **Cel** — co weryfikujemy.
2. **Dane wejściowe (given)** — wymiary, parametry, dystrybucje.
3. **Co robimy (steps)** — algorytm dopasowania i ewaluacji.
4. **Co dostajemy (outputs)** — printy, wykresy, assercje.
5. **Oczekiwane wyniki** — wartości referencyjne do interpretacji.

**Identyfikowalność:** `W` i `Z` są w FA tylko z dokładnością do permutacji kolumn i znaków (każda macierz obrotowa `A` daje `(WA)(A⁻¹Z) = WZ`). Wszystkie porównania `W_true` ↔ `W_est` wykonujemy po **tej samej permutacji Hungarian**, którą wyznaczamy z dopasowania `Z`.

**Macierz S (spike-and-slab):** w `ARD_SS` ładunek faktoryzuje się jako `W_{d,k} = S_{d,k} · Ŵ_{d,k}`, gdzie `S_{d,k} ∈ {0,1}` to binarny wskaźnik selekcji cechy. W wariacyjnej aproksymacji `S` ma rozkład Bernoulliego, a `E[S_{d,k}]` (czyli `s_m_node.vi_gamma`) to prawdopodobieństwo, że dany ładunek jest niezerowy.

### Importy

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import roc_auc_score

from src import FACTModel
from src.enums import Likelihood, WPrior, ZPrior
from src.model_config import CohortPriorConfig, ModelConfig, SimpleViewConfig
from src.views import Views

### Helpery: generacja danych ze znanym W, fitting, permutacja, odtworzenie

In [ ]:
def generate_data_with_known_W(mu_ck, sigma, W_list, n_per_cohort, noise=1.0, seed=0):
    """Generuje Y_m = Z W_m^T + szum przy ustalonych macierzach W_list.

    mu_ck   : (C, K) macierz średnich kohortowych czynników
    sigma   : skalar lub (C, K) — odchylenia std per kohort per czynnik
    W_list  : lista macierzy (D_m, K) — prawdziwe ładunki per widok
    n_per_cohort: lista rozmiarów kohort
    """
    rng = np.random.default_rng(seed)
    C, K = mu_ck.shape
    codes = np.repeat(np.arange(C), n_per_cohort)
    Z = np.array([rng.normal(mu_ck[c], sigma) for c in codes])
    Ys = [
        Z @ W.T + rng.normal(scale=noise, size=(Z.shape[0], W.shape[0]))
        for W in W_list
    ]
    cohorts = np.array([f'c{c}' for c in codes])
    return Views.from_list(Ys, cohorts=cohorts), Z, codes


def fit_ard(views, K, max_iter=150, seed=0):
    """Fit z WPrior.ARD i ZPrior.STD_NORMAL — do SC1 (podstawowa odzysk W)."""
    cfg = ModelConfig(
        simple_view_configs=[
            SimpleViewConfig(likelihood=Likelihood.NORMAL, w_prior=WPrior.ARD)
            for _ in range(views.num_simple)
        ],
        structured_view_configs=[],
        z_priors=[ZPrior.STD_NORMAL] * K,
    )
    m = FACTModel(views=views, K=K, model_config=cfg, seed=seed)
    m.fit(max_iter=max_iter, pretrain=True, elbo_tres=0.0)
    return m


def fit_ard_ss(views, K, max_iter=150, seed=0):
    """Fit z WPrior.ARD_SS i ZPrior.STD_NORMAL — do SC2 (rzadka struktura W)."""
    cfg = ModelConfig(
        simple_view_configs=[
            SimpleViewConfig(likelihood=Likelihood.NORMAL, w_prior=WPrior.ARD_SS)
            for _ in range(views.num_simple)
        ],
        structured_view_configs=[],
        z_priors=[ZPrior.STD_NORMAL] * K,
    )
    m = FACTModel(views=views, K=K, model_config=cfg, seed=seed)
    m.fit(max_iter=max_iter, pretrain=True, elbo_tres=0.0)
    return m


def fit_cohort(views, K, pi=0.5, max_iter=150, seed=0):
    """Fit z WPrior.ARD_SS i ZPrior.COHORT."""
    cfg = ModelConfig(
        simple_view_configs=[
            SimpleViewConfig(likelihood=Likelihood.NORMAL, w_prior=WPrior.ARD_SS)
            for _ in range(views.num_simple)
        ],
        structured_view_configs=[],
        z_priors=[ZPrior.COHORT] * K,
        cohort_prior_config=CohortPriorConfig(pi=pi),
    )
    m = FACTModel(views=views, K=K, model_config=cfg, seed=seed)
    m.fit(max_iter=max_iter, pretrain=True, elbo_tres=0.0)
    return m


def fit_baseline(views, K, max_iter=150, seed=0):
    """Fit z WPrior.ARD_SS i ZPrior.STD_NORMAL — baseline dla SC4."""
    cfg = ModelConfig(
        simple_view_configs=[
            SimpleViewConfig(likelihood=Likelihood.NORMAL, w_prior=WPrior.ARD_SS)
            for _ in range(views.num_simple)
        ],
        structured_view_configs=[],
        z_priors=[ZPrior.STD_NORMAL] * K,
    )
    m = FACTModel(views=views, K=K, model_config=cfg, seed=seed)
    m.fit(max_iter=max_iter, pretrain=True, elbo_tres=0.0)
    return m


def hungarian(Z_true, Z_hat):
    """Permutacja i znaki: Z_hat[:, perm]*signs ≈ Z_true (maks. sumy |corr|)."""
    K = Z_true.shape[1]
    corr = np.array([
        [np.corrcoef(Z_true[:, i], Z_hat[:, j])[0, 1] for j in range(K)]
        for i in range(K)
    ])
    row, col = linear_sum_assignment(-np.abs(corr))
    signs = np.sign(corr[row, col])
    signs[signs == 0] = 1
    return col, signs.astype(float)


def recover_W(W_true, W_est, perm, signs):
    """Zwraca |corr(W_true[:,k], W_est[:,perm[k]]*signs[k])| per czynnik."""
    K = W_true.shape[1]
    return np.array([
        abs(np.corrcoef(W_true[:, k], W_est[:, perm[k]] * signs[k])[0, 1])
        for k in range(K)
    ])


def evaluate_W(model, Z_true, W_true_list):
    """Hungarian na Z, potem |corr| na W per czynnik per widok.

    Zwraca: perm, signs, lista tablic |corr| (po jednej na widok).
    """
    Z_hat = model.get_latent_factors()
    perm, signs = hungarian(Z_true, Z_hat)
    W_corrs = [
        recover_W(W_true, model.fa.nodelist_w[m].E_w, perm, signs)
        for m, W_true in enumerate(W_true_list)
    ]
    return perm, signs, W_corrs

---
## SC1 — Podstawowe odtworzenie W (sweep po poziomie szumu)

**Cel:** Potwierdzenie, że model odzyskuje losowe W_true gdy Z nie ma struktury kohortowej, oraz że jakość odtworzenia spada wraz z rosnącym szumem.

### Dane wejściowe (given)
- `K=3` czynników, `D=20` cech (1 widok), `N=200` próbek, `C=1` kohorta (brak struktury kohortowej)
- `W_true ~ N(0, 1/√K)` — losowa, lecz **ustalona** macierz ładunków (D×K), generowana raz z `seed=42`
- `Z ~ N(0, I_K)` — losowane przy każdej generacji danych
- `Y = Z · W_true.T + ε`, gdzie `ε ~ N(0, noise²)`
- Sweep: `noise ∈ {0.2, 0.5, 1.0, 2.0}` × `seed ∈ {0,1,2,3,4}` = 20 dopasowań
- Model: `WPrior.ARD`, `ZPrior.STD_NORMAL`, `max_iter=150`, `pretrain=True`

### Co robimy (steps)
1. Dla każdej kombinacji (noise, seed):
   a. wygeneruj `Y` z **tą samą** `W_true` ale nowym `Z` i nową realizacją szumu;
   b. dopasuj model (`fit_ard`);
   c. wyciągnij `Z_hat = model.get_latent_factors()` i `W_est = model.fa.nodelist_w[0].E_w`;
   d. dopasuj kolumny algorytmem **węgierskim** na bazie `Z` — uzyskaj `perm`, `signs`;
   e. policz `|corr(W_true[:,k], W_est[:,perm[k]] * signs[k])|` per czynnik, uśrednij po `k`.
2. Dla `seed=0` zapisz przepermutowane `W_est` — do późniejszej heatmapy.
3. Zbierz statystyki (mean, std) per poziom szumu.

### Co dostajemy (outputs)
- **Print** per noise: `mean|corr| ± std` z 5 ziaren.
- **Wykres 1** (errorbar): średnie `|corr|` vs `noise`, próg `0.75` zaznaczony linią przerywaną.
- **Wykres 2** (heatmapy): `W_true` obok `W_est` dla każdego poziomu szumu (5 paneli) — pokazuje, jak ARD coraz mocniej kurczy kolumny przy rosnącym szumie.
- **Assercje:**
  - `means_sc1[0] > 0.75` — przy najniższym szumie odzysk musi przekraczać próg jakości.
  - `means_sc1[-1] < means_sc1[0]` — odzysk spada wraz ze szumem.

### Oczekiwane wyniki
- `noise=0.2`: `mean|corr| ≈ 0.80` (bliskie ideału z uwzględnieniem nieidentyfikowalności obrotowej FA).
- Monotoniczny spadek do `≈0.74` przy `noise=2.0`.
- Na heatmapach: przy niskim szumie wyraźna struktura `W_true`, przy wysokim — kolumny `W_est` blakną (ARD shrinkage).

In [ ]:
rng_sc1 = np.random.default_rng(42)
K_sc1, D_sc1, N_sc1 = 3, 20, 200
W_true_sc1 = rng_sc1.normal(size=(D_sc1, K_sc1)) / np.sqrt(K_sc1)  # (D, K)

# C=1: brak struktury kohortowej — mu_ck = 0
mu_sc1 = np.zeros((1, K_sc1))

# Zatrzymujemy się na noise=2.0: przy noise=5.0 ARD kolapsuje W do zera,
# co powoduje nieznaczące korelacje numeryczne w corrcoef.
noise_levels = [0.2, 0.5, 1.0, 2.0]
seeds_sc1 = [0, 1, 2, 3, 4]
results_sc1 = {nl: [] for nl in noise_levels}
models_sc1 = {}  # zapisany model per noise (seed=0) — do późniejszej wizualizacji W

for noise in noise_levels:
    for seed in seeds_sc1:
        views, Z_true, _ = generate_data_with_known_W(
            mu_sc1, sigma=1.0, W_list=[W_true_sc1],
            n_per_cohort=[N_sc1], noise=noise, seed=seed
        )
        model = fit_ard(views, K=K_sc1, seed=seed)
        perm, signs, W_corrs = evaluate_W(model, Z_true, [W_true_sc1])
        results_sc1[noise].append(W_corrs[0].mean())
        if seed == 0:
            W_est = model.fa.nodelist_w[0].E_w[:, perm] * signs[None, :]
            models_sc1[noise] = W_est  # permutowane W_est dla wizualizacji
    print(f"noise={noise:.1f}  mean|corr|={np.mean(results_sc1[noise]):.3f} "
          f"± {np.std(results_sc1[noise]):.3f}")

In [ ]:
means_sc1 = [np.mean(results_sc1[nl]) for nl in noise_levels]
stds_sc1  = [np.std(results_sc1[nl])  for nl in noise_levels]

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(noise_levels, means_sc1, yerr=stds_sc1, marker='o', linewidth=2,
            capsize=4, color='steelblue', label='mean |corr(W_true, W_est)|')
ax.axhline(0.75, color='gray', linestyle='--', alpha=0.6, label='próg 0.75')
ax.set_xlabel('Poziom szumu')
ax.set_ylabel('|corr| (W_true vs W_est)')
ax.set_title('SC1 — Odtworzenie W vs poziom szumu')
ax.set_ylim([0, 1.05])
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

assert means_sc1[0] > 0.75, f"SC1 FAIL: przy noise=0.2 mean|corr|={means_sc1[0]:.3f} < 0.75"
assert means_sc1[-1] < means_sc1[0], "SC1 FAIL: odtworzenie nie spada wraz z szumem"
print("SC1 PASS — odtworzenie W monotonicznie maleje ze szumem, próg 0.75 spełniony przy niskim szumie.")

In [ ]:
# SC1 — heatmapa W_true vs W_est dla każdego poziomu szumu
vmax_sc1 = np.abs(W_true_sc1).max()
fig, axes = plt.subplots(1, len(noise_levels) + 1, figsize=(3 * (len(noise_levels) + 1), 5))

kw = dict(cmap='coolwarm', center=0, vmin=-vmax_sc1, vmax=vmax_sc1,
          xticklabels=[f'Z{k}' for k in range(K_sc1)],
          yticklabels=False, cbar=False)

sns.heatmap(W_true_sc1, ax=axes[0], **kw)
axes[0].set_title('W_true')
axes[0].set_ylabel('Cechy')

for i, noise in enumerate(noise_levels):
    sns.heatmap(models_sc1[noise], ax=axes[i + 1], **kw)
    axes[i + 1].set_title(f'W_est, noise={noise}\n|corr|={results_sc1[noise][0]:.2f}')

plt.suptitle('SC1 — Degradacja odtworzenia W wraz ze szumem (seed=0)')
plt.tight_layout()
plt.show()

---
## SC2 — Odtworzenie rzadkiej struktury blokowej W (ARD_SS)

**Cel:** Sprawdzenie, czy spike-and-slab (`ARD_SS`) poprawnie identyfikuje **które cechy** ładują się na **które czynniki**, gdy prawdziwa struktura jest rzadka i blokowa.

### Dane wejściowe (given)
- `K=3` czynników, `D=15` cech podzielonych na 3 bloki po 5, `N=200` próbek, `C=1` kohorta
- `W_true` blokdiagonalna:
  - `W_true[0:5,  0] = 1.0`, reszta kolumny 0 = 0
  - `W_true[5:10, 1] = 1.0`, reszta kolumny 1 = 0
  - `W_true[10:15,2] = 1.0`, reszta kolumny 2 = 0
- `Z ~ N(0, I_K)`, `noise=0.5`, `seed=7` (jedno dopasowanie)
- Model: `WPrior.ARD_SS`, `ZPrior.STD_NORMAL`

### Co robimy (steps)
1. Wygeneruj `Y = Z · W_true.T + szum`.
2. Dopasuj model (`fit_ard_ss`).
3. Hungarian na `Z` → `perm`, `signs`.
4. Wyciągnij dwa obiekty z węzła W:
   - `E[W] = model.fa.nodelist_w[0].E_w` (D×K) — wartość ładunków;
   - `E[S] = model.fa.nodelist_w[0].s_m_node.vi_gamma` (D×K) — prawdopodobieństwo, że dany ładunek jest niezerowy (spike-slab).
5. Spermutuj `E[W]` i `E[S]` zgodnie z `perm` × `signs` (dla E[S] tylko permutacja, bo to wielkość ≥ 0).
6. Zdefiniuj prawdziwy wzorzec rzadkości `S_true = (W_true != 0)`.
7. Metryki:
   - `|corr(W_true[:,k], W_est[:,k])|` per czynnik;
   - **ROC AUC** dla każdej kolumny: traktuj `E[S][:,k]` jako miękki score klasyfikatora niezerowości względem `S_true[:,k]`.

### Co dostajemy (outputs)
- **Print:** permutacja, `|corr|` per czynnik, mean `|corr|`; AUC per czynnik, mean AUC.
- **Wykres** (4 panele heatmap): `W_true | W_est | S_true | E[S]` — widać blokową strukturę w obu wariantach.
- **Assercje:**
  - `mean|corr| > 0.85` — kolumny `W` poprawnie odtworzone;
  - `mean AUC > 0.85` — spike-and-slab dyskryminuje niezerowe ładunki od zerowych.

### Oczekiwane wyniki
- Heatmapa `W_est` powtarza strukturę bloków `W_true`, choć z drobnymi przeciekami między blokami.
- `E[S]` jest niebieska (≈1) wewnątrz bloków, bliska 0 poza nimi.
- `|corr|` per czynnik: ≈ `[0.95, 0.92, 0.88]`, AUC: `[1.00, 0.75, 0.85]` — mocne czynniki AUC=1.0, słabsze ~0.8.

In [ ]:
K_sc2, D_sc2 = 3, 15
block = D_sc2 // K_sc2  # = 5

# Blokdiagonalne W_true
W_true_sc2 = np.zeros((D_sc2, K_sc2))
for k in range(K_sc2):
    W_true_sc2[k * block:(k + 1) * block, k] = 1.0

mu_sc2 = np.zeros((1, K_sc2))
views_sc2, Z_true_sc2, _ = generate_data_with_known_W(
    mu_sc2, sigma=1.0, W_list=[W_true_sc2],
    n_per_cohort=[200], noise=0.5, seed=7
)

model_sc2 = fit_ard_ss(views_sc2, K=K_sc2, seed=7)
perm_sc2, signs_sc2, W_corrs_sc2 = evaluate_W(model_sc2, Z_true_sc2, [W_true_sc2])

print("Permutacja:", perm_sc2)
print("|corr| per czynnik:", W_corrs_sc2[0].round(3))
print("Mean |corr|:", W_corrs_sc2[0].mean().round(3))

In [ ]:
# Wzorzec rzadkości: E[S] z modelu vs S_true z danych
W_est_sc2 = model_sc2.fa.nodelist_w[0].E_w                    # (D, K)
S_est_sc2 = model_sc2.fa.nodelist_w[0].s_m_node.vi_gamma      # (D, K)
S_true_sc2 = (W_true_sc2 != 0).astype(float)                  # (D, K)

# Permutuj kolumny E[W] i E[S] żeby odpowiadały kolumnom W_true
W_est_perm = W_est_sc2[:, perm_sc2] * signs_sc2[None, :]
S_est_perm = S_est_sc2[:, perm_sc2]

# ROC AUC: czy E[S] dyskryminuje prawdziwe niezerowe ładunki?
auc_per_k = [
    roc_auc_score(S_true_sc2[:, k], S_est_perm[:, k])
    for k in range(K_sc2)
]
print("AUC per czynnik:", [round(a, 3) for a in auc_per_k])
print("Mean AUC:", round(np.mean(auc_per_k), 3))

# Wizualizacja: W_true | W_est | S_true | E[S]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

kw = dict(cmap='coolwarm', center=0, vmin=-1.2, vmax=1.2,
          xticklabels=[f'Z{k}' for k in range(K_sc2)],
          yticklabels=False)
sns.heatmap(W_true_sc2, ax=axes[0], **kw)
axes[0].set_title('W_true')

sns.heatmap(W_est_perm, ax=axes[1], **kw)
axes[1].set_title('W_est (permutowane)')

kw_bin = dict(cmap='Blues', vmin=0, vmax=1,
              xticklabels=[f'Z{k}' for k in range(K_sc2)],
              yticklabels=False)
sns.heatmap(S_true_sc2, ax=axes[2], **kw_bin)
axes[2].set_title('S_true (niezerowe W)')

sns.heatmap(S_est_perm, ax=axes[3], **kw_bin)
axes[3].set_title('E[S] (permutowane)')

plt.suptitle('SC2 — Blokdiagonalna struktura W: prawdziwa vs estymowana')
plt.tight_layout()
plt.show()

assert W_corrs_sc2[0].mean() > 0.85, f"SC2 FAIL: mean|corr|W={W_corrs_sc2[0].mean():.3f} < 0.85"
assert np.mean(auc_per_k) > 0.85, f"SC2 FAIL: mean AUC={np.mean(auc_per_k):.3f} < 0.85"
print("SC2 PASS — blokowa rzadkość W odtworzona, AUC > 0.85 i |corr|W > 0.85.")

---
## SC3 — Detekcja cechy różnicującej kohorty (główny pomysł użytkownika)

**Cel:** Skonstruować scenariusz, w którym **jedna cecha** (feature 0) ładuje się *wyłącznie* na jeden czynnik (Z0), a Z0 z kolei jest jedynym czynnikiem różnicującym dwie kohorty. Wówczas model powinien:
(a) wykryć Z0 jako czynnik kohortowy (`E[γ] → 1`);
(b) poprawnie umieścić maksimum ładunku `W[:,0]` na cechę 0;
(c) odtworzyć całą kolumnę `W_true[:,0]` z wysoką korelacją.

### Dane wejściowe (given)
- `C=2` kohorty po `N=100` próbek, `K=3` czynniki, `D=15` cech (1 widok)
- `W_true`:
  - tło: `W_true ~ N(0, 0.15²)` dla wszystkich (d, k);
  - kolumna 0 wyzerowana → `W_true[:, 0] = 0`;
  - pojedynczy unikalny ładunek: `W_true[0, 0] = 2.0` (cecha 0 ↔ tylko Z0).
- `mu_ck` (średnie czynników per kohorta):
  - kohorta 0: `[+3.0, 0.0, 0.0]` — Z0 silnie przesunięty;
  - kohorta 1: `[0.0, 0.0, 0.0]` — bez przesunięcia.
- `Z_{n,k} ~ N(mu_ck[c(n), k], 1)`, `noise=0.5`, `seed=13`
- Model: `WPrior.ARD_SS`, `ZPrior.COHORT`, `pi=0.5`

### Co robimy (steps)
1. Skonstruuj `W_true` i `mu_ck` jak wyżej.
2. Wygeneruj `Y` przez `generate_data_with_known_W`.
3. **Sanity-check danych:** policz średnią cechy 0 w każdej kohorcie — powinna być znacznie wyższa w kohorcie 0 (efekt `W_true[0,0]·E[Z0|c=0] ≈ 2·3 = 6`).
4. Dopasuj model (`fit_cohort`).
5. Hungarian na `Z` → `perm`, `signs`. Niech `k0 = perm[0]` to indeks kolumny `W_est` dopasowanej do prawdziwego Z0.
6. Wyciągnij `W_col0 = model.fa.nodelist_w[0].E_w[:, k0] * signs[0]`.
7. Wyznacz **cechę pikową**: `argmax(|W_col0|)`. Powinna wynosić `0`.
8. Wyciągnij macierz `E[γ]` per kohort × czynnik: `np.column_stack([priors[k].E_gamma for k])` i spermutuj.
9. Oblicz `|corr|` per czynnik (jak w SC1/SC2).

### Co dostajemy (outputs)
- **Print:**
  - średnie cechy 0 per kohorta (sanity-check danych);
  - permutacja, `|corr|` per czynnik, indeks cechy pikowej, tabela `E[γ]` (C × K).
- **Wykres 1** (3 panele):
  - violin: rozkład cechy 0 w kohorcie 0 vs 1 (potwierdza, że sygnał istnieje w surowych danych);
  - bar: `W_col0_est` po cechach, cecha 0 czerwona, reszta niebieska — powinien być wyraźny pik na 0;
  - heatmapa `E[γ]` (C × K) — Z0 niebieskie, Z1/Z2 białe.
- **Wykres 2** (pełna heatmapa W): `W_true | W_est | E[S]` — wszystkie 15 cech × 3 czynniki, pokazuje że tylko `(f0, Z0)` ma silny niebieski/czerwony punkt.
- **Assercje:**
  - `peak_feature == 0` — najwyższy |W| czynnika kohortowego jest na cesze 0;
  - `|corr|W[Z0] > 0.8` — kolumna ładunków poprawnie odzyskana;
  - `mean E[γ] na Z0 > 0.5` — czynnik wykryty jako kohortowy.

### Oczekiwane wyniki
- Średnia cechy 0: kohorta 0 ≈ 6.0, kohorta 1 ≈ 0.0 (różnica ≈ 6 z `W_true[0,0]·μ`).
- `|corr|W[Z0] ≈ 0.999` (prawie idealne odtworzenie tej konkretnej kolumny).
- Bar plot: czerwony słupek na cesze 0 dominuje wyraźnie nad resztą (~10× wyższy).
- `E[γ][:, Z0] ≈ 1.0`, `E[γ][:, Z1/Z2] ≈ 0.0`.

In [ ]:
rng_sc3 = np.random.default_rng(13)
K_sc3, D_sc3 = 3, 15

# W_true: cecha 0 obciąża wyłącznie czynnik 0; pozostałe cechy mają małe, gęste ładunki
W_true_sc3 = rng_sc3.normal(size=(D_sc3, K_sc3)) * 0.15  # małe tło dla pozostałych cech
W_true_sc3[:, 0] = 0.0          # wyzeruj kolumnę czynnika 0
W_true_sc3[0, 0] = 2.0          # cecha 0 ładuje się wyłącznie na czynnik 0

# Kohorty: tylko czynnik 0 ma sygnał kohortowy
mu_sc3 = np.array([[3.0, 0.0, 0.0],   # kohorta 0: wysoki Z_0
                   [0.0, 0.0, 0.0]])   # kohorta 1: brak przesunięcia

views_sc3, Z_true_sc3, codes_sc3 = generate_data_with_known_W(
    mu_sc3, sigma=1.0, W_list=[W_true_sc3],
    n_per_cohort=[100, 100], noise=0.5, seed=13
)

# Sprawdzenie generacji danych: rozkład cechy 0 w obu kohortach powinien się różnić
Y0 = views_sc3.simple[0].data[:, 0]
c0_mask = codes_sc3 == 0
print(f"Feature 0 — kohorta 0: mean={Y0[c0_mask].mean():.2f}, "
      f"kohorta 1: mean={Y0[~c0_mask].mean():.2f}  (oczekiwana duża różnica)")

In [ ]:
model_sc3 = fit_cohort(views_sc3, K=K_sc3, pi=0.5, seed=13)

perm_sc3, signs_sc3, W_corrs_sc3 = evaluate_W(model_sc3, Z_true_sc3, [W_true_sc3])
print(f"Permutacja: {perm_sc3}")
print(f"|corr| W per czynnik: {W_corrs_sc3[0].round(3)}")

# Czynnik kohortowy (Z0) po dopasowaniu
k0_est = perm_sc3[0]  # która kolumna W_est odpowiada Z0
W_col0_est = model_sc3.fa.nodelist_w[0].E_w[:, k0_est] * signs_sc3[0]

# Test 1: cecha 0 powinna mieć najwyższy ładunek na czynnik 0
peak_feature = np.argmax(np.abs(W_col0_est))
print(f"\nNajwyższy |W| na czynnik kohortowy (Z0): cecha {peak_feature}  (oczekiwana: 0)")

# Test 2: E[gamma] — czynnik 0 powinien być wykryty jako kohortowy
priors = model_sc3.fa.node_z.z_priors
E_gamma = np.column_stack([priors[k].E_gamma for k in range(K_sc3)])  # (C, K)
E_gamma_perm = E_gamma[:, perm_sc3]
print(f"\nE[gamma] (kohort x czynniki, po permutacji):")
print(pd.DataFrame(E_gamma_perm.round(3), 
                   index=['c0','c1'], 
                   columns=['Z0 (kohortowy)', 'Z1', 'Z2']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel 1: violin rozkładu cechy 0 per kohorta (weryfikacja danych)
violin_data = [Y0[c0_mask], Y0[~c0_mask]]
axes[0].violinplot(violin_data, positions=[0, 1])
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Kohorta 0', 'Kohorta 1'])
axes[0].set_ylabel('Wartość cechy 0')
axes[0].set_title('Dane: cecha 0 per kohorta\n(oczekiwana duża różnica)')
axes[0].grid(alpha=0.3)

# Panel 2: W[:,k0] — ładunki czynnika kohortowego
colors = ['tomato' if i == 0 else 'steelblue' for i in range(D_sc3)]
axes[1].bar(range(D_sc3), W_col0_est, color=colors)
axes[1].set_xlabel('Indeks cechy')
axes[1].set_ylabel('W_est')
axes[1].set_title('SC3 — Ładunki czynnika kohortowego Z0\n(cecha 0 zaznaczona na czerwono)')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].grid(alpha=0.3)

# Panel 3: heatmapa E[gamma] (kohorty x czynniki)
sns.heatmap(E_gamma_perm, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
            xticklabels=['Z0 (kohortowy)', 'Z1', 'Z2'],
            yticklabels=['c0', 'c1'], ax=axes[2])
axes[2].set_title('E[gamma]: aktywacja kohortowa\n(oczekiwana: Z0≈1, Z1,Z2≈0)')

plt.suptitle('SC3 — Detekcja cechy różnicującej kohorty')
plt.tight_layout()
plt.show()

assert peak_feature == 0, f"SC3 FAIL: najwyższy |W| czynnika kohortowego na cesze {peak_feature}, nie 0"
assert W_corrs_sc3[0][0] > 0.8, f"SC3 FAIL: |corr|W dla Z0 = {W_corrs_sc3[0][0]:.3f} < 0.8"
assert E_gamma_perm[:, 0].mean() > 0.5, "SC3 FAIL: E[gamma] na Z0 nie wskazuje aktywacji kohortowej"
print("SC3 PASS — cecha 0 poprawnie zidentyfikowana jako unikalna dla czynnika kohortowego, "
      "E[gamma] aktywuje się na Z0.")

In [ ]:
# SC3 — pełna heatmapa W_true vs W_est (wszystkie czynniki)
W_est_sc3 = model_sc3.fa.nodelist_w[0].E_w[:, perm_sc3] * signs_sc3[None, :]
S_est_sc3 = model_sc3.fa.nodelist_w[0].s_m_node.vi_gamma[:, perm_sc3]
vmax_sc3 = max(np.abs(W_true_sc3).max(), np.abs(W_est_sc3).max())

fig, axes = plt.subplots(1, 3, figsize=(12, 5))

kw = dict(cmap='coolwarm', center=0, vmin=-vmax_sc3, vmax=vmax_sc3,
          xticklabels=['Z0 (kohortowy)', 'Z1', 'Z2'],
          yticklabels=[f'f{i}' for i in range(D_sc3)])
sns.heatmap(W_true_sc3, ax=axes[0], **kw)
axes[0].set_title('W_true\n(f0 ładuje wyłącznie na Z0)')

sns.heatmap(W_est_sc3, ax=axes[1], **kw)
axes[1].set_title(f'W_est (permutowane)\n|corr| Z0={W_corrs_sc3[0][0]:.2f}')

sns.heatmap(S_est_sc3, ax=axes[2], cmap='Blues', vmin=0, vmax=1,
            xticklabels=['Z0 (kohortowy)', 'Z1', 'Z2'],
            yticklabels=[f'f{i}' for i in range(D_sc3)])
axes[2].set_title('E[S] (sparsity probability)\nczy cecha ładuje na czynnik?')

plt.suptitle('SC3 — Pełna macierz ładunków W: prawdziwa, estymowana i wzorzec rzadkości')
plt.tight_layout()
plt.show()

---
## SC4 — Odporność odtworzenia W przy rosnącym sygnale kohortowym (cohort vs baseline)

**Cel:** Sprawdzić, czy model kohortowy (`ZPrior.COHORT`) utrzymuje jakość odtworzenia `W` gdy sygnał kohortowy się wzmacnia, a baseline (`ZPrior.STD_NORMAL`) — który zakłada `Z ~ N(0, I)` — może się degradować, bo przesunięcia kohortowe wchodzą do estymacji `Z` jako zewnętrzny bias.

### Dane wejściowe (given)
- `C=2` kohorty po `N=100`, `K=3`, `D=15` cech (1 widok)
- `W_true ~ N(0, 1/√K)` — **ustalona** (jedna realizacja z `seed=99`), gęsta, bez struktury kohortowej w `W`
- `mu_ck = [[mu_0, 0, 0], [0, 0, 0]]` — tylko czynnik Z0 przesunięty w kohorcie 0; sweep `mu_0 ∈ {0, 1, 2, 3, 4}`
- 5 ziaren danych (`seeds=[10..14]`) na każdy poziom `mu_0`
- `noise=0.5`
- Dwa modele równolegle:
  - `fit_cohort`: `WPrior.ARD_SS + ZPrior.COHORT + pi=0.5`;
  - `fit_baseline`: `WPrior.ARD_SS + ZPrior.STD_NORMAL`.

### Co robimy (steps)
1. Dla każdego `mu_0`:
   a. dla każdego `seed`: wygeneruj `Y`, dopasuj **oba** modele;
   b. Hungarian + `|corr|` per czynnik dla każdego z nich;
   c. uśrednij `|corr|` po czynnikach → jedna liczba per (mu_0, seed, model).
2. Przy `mu_0 = 4` (maksymalny sygnał) i pierwszym seedzie zapisz przepermutowane `W_est` z obu modeli — do późniejszej heatmapy.
3. Zagregowane statystyki: mean ± std per (mu_0, model).
4. Policz różnicę `cohort - baseline` przy `mu_0_max` — sygnalizuje, czy model kohortowy wygrywa, dorównuje, czy przegrywa.

### Co dostajemy (outputs)
- **Print** per mu_0: `cohort=...` vs `baseline=...` średnie `|corr|`.
- **Wykres 1** (errorbar, 2 linie): `|corr|W` vs `mu_0`, niebieska linia = CohortFACTM, koralowa = baseline.
- **Wykres 2** (3 panele heatmap przy `mu_0=4`): `W_true | W_est cohort | W_est baseline` — pokazuje wizualnie, czy oba modele odtwarzają tę samą strukturę W.
- **Print** finalny: różnica `|corr|` przy największym sygnale kohortowym.
- **Brak twardych assercji** — to test odporności, nie binarny pass/fail; wynik interpretujemy jakościowo.

### Oczekiwane wyniki
- Oba modele utrzymują `|corr| ≈ 0.83–0.85` w całym zakresie `mu_0`.
- W okolicy `mu_0 ∈ {1, 2}` model kohortowy może wygrywać kilka setnych (~`0.85` vs `0.79`), bo baseline zaczyna „przeznaczać" jeden czynnik na odzwierciedlenie średniej kohortowej zamiast na strukturę `W`.
- Przy `mu_0 = 4` różnica znika lub jest minimalna (~`±0.005`) — przy bardzo silnym sygnale baseline również znajduje rozwiązanie zdominowane przez Z0 cohort-shift.
- Na heatmapach `W_est` obu modeli powinny być wizualnie podobne do `W_true` (modulo nieidentyfikowalność obrotowa).

In [ ]:
rng_sc4 = np.random.default_rng(99)
K_sc4, D_sc4 = 3, 15
W_true_sc4 = rng_sc4.normal(size=(D_sc4, K_sc4)) / np.sqrt(K_sc4)

mu0_values = [0.0, 1.0, 2.0, 3.0, 4.0]
seeds_sc4 = [10, 11, 12, 13, 14]

results_sc4_cohort   = {mu0: [] for mu0 in mu0_values}
results_sc4_baseline = {mu0: [] for mu0 in mu0_values}
W_est_sc4_cohort   = {}  # zapisany W_est dla najwyższego mu0 (do heatmapy)
W_est_sc4_baseline = {}

for mu0 in mu0_values:
    mu_sc4 = np.array([[mu0, 0.0, 0.0],
                       [0.0, 0.0, 0.0]])
    for seed in seeds_sc4:
        views, Z_true, _ = generate_data_with_known_W(
            mu_sc4, sigma=1.0, W_list=[W_true_sc4],
            n_per_cohort=[100, 100], noise=0.5, seed=seed
        )
        m_c = fit_cohort(views, K=K_sc4, pi=0.5, seed=seed)
        m_b = fit_baseline(views, K=K_sc4, seed=seed)

        perm_c, signs_c, wc_c = evaluate_W(m_c, Z_true, [W_true_sc4])
        perm_b, signs_b, wc_b = evaluate_W(m_b, Z_true, [W_true_sc4])

        results_sc4_cohort[mu0].append(wc_c[0].mean())
        results_sc4_baseline[mu0].append(wc_b[0].mean())

        # Zapisz reprezentatywne W_est przy najsilniejszym sygnale kohortowym
        if mu0 == mu0_values[-1] and seed == seeds_sc4[0]:
            W_est_sc4_cohort[mu0]   = m_c.fa.nodelist_w[0].E_w[:, perm_c] * signs_c[None, :]
            W_est_sc4_baseline[mu0] = m_b.fa.nodelist_w[0].E_w[:, perm_b] * signs_b[None, :]

    print(f"mu0={mu0:.1f}  cohort={np.mean(results_sc4_cohort[mu0]):.3f}  "
          f"baseline={np.mean(results_sc4_baseline[mu0]):.3f}")

In [ ]:
means_c  = [np.mean(results_sc4_cohort[mu0])   for mu0 in mu0_values]
stds_c   = [np.std(results_sc4_cohort[mu0])    for mu0 in mu0_values]
means_b  = [np.mean(results_sc4_baseline[mu0]) for mu0 in mu0_values]
stds_b   = [np.std(results_sc4_baseline[mu0])  for mu0 in mu0_values]

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(mu0_values, means_c, yerr=stds_c, marker='o', linewidth=2,
            capsize=4, color='steelblue', label='CohortFACTM')
ax.errorbar(mu0_values, means_b, yerr=stds_b, marker='s', linewidth=2,
            capsize=4, color='coral', label='Baseline (STD_NORMAL)')
ax.set_xlabel('Siła sygnału kohortowego μ_0')
ax.set_ylabel('Mean |corr(W_true, W_est)|')
ax.set_title('SC4 — Odporność odtworzenia W: Cohort vs Baseline')
ax.set_ylim([0, 1.05])
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Przy dużym sygnale kohortowym model kohortowy powinien dorównywać lub bić baseline
gap_at_max = means_c[-1] - means_b[-1]
print(f"\nRóżnica |corr|W przy mu0={mu0_values[-1]}: cohort - baseline = {gap_at_max:+.3f}")
print("SC4 — wyniki pokazują odporność modelu kohortowego na rosnący sygnał kohortowy.")

In [ ]:
# SC4 — heatmapa W_true vs W_est (cohort vs baseline) przy największym sygnale kohortowym
mu0_max = mu0_values[-1]
W_c = W_est_sc4_cohort[mu0_max]
W_b = W_est_sc4_baseline[mu0_max]
vmax_sc4 = max(np.abs(W_true_sc4).max(), np.abs(W_c).max(), np.abs(W_b).max())

fig, axes = plt.subplots(1, 3, figsize=(11, 5))

kw = dict(cmap='coolwarm', center=0, vmin=-vmax_sc4, vmax=vmax_sc4,
          xticklabels=[f'Z{k}' for k in range(K_sc4)],
          yticklabels=False)

sns.heatmap(W_true_sc4, ax=axes[0], **kw)
axes[0].set_title('W_true')
axes[0].set_ylabel('Cechy')

sns.heatmap(W_c, ax=axes[1], **kw)
axes[1].set_title(f'W_est (CohortFACTM)\n|corr|={results_sc4_cohort[mu0_max][0]:.2f}')

sns.heatmap(W_b, ax=axes[2], **kw)
axes[2].set_title(f'W_est (Baseline)\n|corr|={results_sc4_baseline[mu0_max][0]:.2f}')

plt.suptitle(f'SC4 — Porównanie odzysku W przy μ_0={mu0_max} (seed={seeds_sc4[0]})')
plt.tight_layout()
plt.show()